In [ ]:
!pip uninstall -y transformers peft accelerate bitsandbytes datasets loralib
!pip install --no-cache-dir "transformers>=4.41.0" "peft>=0.10.0" accelerate bitsandbytes datasets loralib

In [ ]:
categories = [
    "ACCOUNT", "CANCELLATION_FEE", "DELIVERY", "FEEDBACK",
    "INVOICE", "NEWSLETTER", "ORDER", "PAYMENT",
    "REFUND", "SHIPPING_ADDRESS"
]

category_entities = {
    "ACCOUNT": ["create_account", "delete_account", "edit_account", "switch_account"],
    "CANCELLATION_FEE": ["check_cancellation_fee"],
    "DELIVERY": ["delivery_options"],
    "FEEDBACK": ["complaint", "review"],
    "INVOICE": ["check_invoice", "get_invoice"],
    "NEWSLETTER": ["newsletter_subscription"],
    "ORDER": ["cancel_order", "change_order", "place_order"],
    "PAYMENT": ["check_payment_methods", "payment_issue"],
    "REFUND": ["check_refund_policy", "track_refund"],
    "SHIPPING_ADDRESS": ["change_shipping_address", "set_up_shipping_address"]
}

entity_to_intents = {
    "Order Number": [
        "cancel_order", "change_order", "change_shipping_address", "check_invoice",
        "check_refund_policy", "complaint", "delivery_options", "delivery_period",
        "get_invoice", "get_refund", "place_order", "track_order", "track_refund"
    ],
    "Invoice Number": ["check_invoice", "get_invoice"],
    "Online Order Interaction": [
        "cancel_order", "change_order", "check_refund_policy", "delivery_period",
        "get_refund", "review", "track_order", "track_refund"
    ],
    "Online Payment Interaction": ["cancel_order", "check_payment_methods"],
    "Online Navigation Step": ["complaint", "delivery_options"],
    "Online Customer Support Channel": [
        "check_refund_policy", "complaint", "contact_human_agent", "delete_account",
        "delivery_options", "edit_account", "get_refund", "payment_issue",
        "registration_problems", "switch_account"
    ],
    "Profile": ["switch_account"],
    "Profile Type": ["switch_account"],
    "Settings": [
        "cancel_order", "change_order", "change_shipping_address", "check_cancellation_fee",
        "check_invoice", "check_payment_methods", "contact_human_agent", "delete_account",
        "delivery_options", "edit_account", "get_invoice", "newsletter_subscription",
        "payment_issue", "place_order", "recover_password", "registration_problems",
        "set_up_shipping_address", "switch_account", "track_order", "track_refund"
    ],
    "Online Company Portal Info": ["cancel_order", "edit_account"],
    "Date": ["check_invoice", "check_refund_policy", "get_refund", "track_order", "track_refund"],
    "Date Range": ["check_cancellation_fee", "check_invoice", "get_invoice"],
    "Shipping Cut-off Time": ["delivery_options"],
    "Delivery City": ["delivery_options"],
    "Delivery Country": ["check_payment_methods", "check_refund_policy", "delivery_options", "review", "switch_account"],
    "Salutation": [
        "cancel_order", "check_payment_methods", "check_refund_policy", "create_account",
        "delete_account", "delivery_options", "get_refund", "recover_password", "review",
        "set_up_shipping_address", "switch_account", "track_refund"
    ],
    "Client First Name": ["check_invoice", "get_invoice"],
    "Client Last Name": ["check_invoice", "create_account", "get_invoice"],
    "Customer Support Phone Number": [
        "change_shipping_address", "contact_customer_service", "contact_human_agent", "payment_issue"
    ],
    "Customer Support Email": [
        "cancel_order", "change_shipping_address", "check_invoice", "check_refund_policy",
        "complaint", "contact_customer_service", "contact_human_agent", "get_invoice",
        "get_refund", "newsletter_subscription", "payment_issue", "recover_password",
        "registration_problems", "review", "set_up_shipping_address", "switch_account"
    ],
    "Live Chat Support": [
        "check_refund_policy", "complaint", "contact_human_agent", "delete_account",
        "delivery_options", "edit_account", "get_refund", "payment_issue", "recover_password",
        "registration_problems", "review", "set_up_shipping_address", "switch_account", "track_order"
    ],
    "Website URL": [
        "check_payment_methods", "check_refund_policy", "complaint", "contact_customer_service",
        "contact_human_agent", "create_account", "delete_account", "delivery_options",
        "get_refund", "newsletter_subscription", "payment_issue", "place_order", "recover_password",
        "registration_problems", "review", "switch_account"
    ],
    "Upgrade Account": ["create_account", "edit_account", "switch_account"],
    "Account Type": [
        "cancel_order", "change_order", "change_shipping_address", "check_cancellation_fee",
        "check_invoice", "check_payment_methods", "check_refund_policy", "complaint",
        "contact_customer_service", "contact_human_agent", "create_account", "delete_account",
        "delivery_options", "delivery_period", "edit_account", "get_invoice", "get_refund",
        "newsletter_subscription", "payment_issue", "place_order", "recover_password",
        "registration_problems", "review", "set_up_shipping_address", "switch_account",
        "track_order", "track_refund"
    ],
    "Account Category": [
        "cancel_order", "change_order", "change_shipping_address", "check_cancellation_fee",
        "check_invoice", "check_payment_methods", "check_refund_policy", "complaint",
        "contact_customer_service", "contact_human_agent", "create_account", "delete_account",
        "delivery_options", "delivery_period", "edit_account", "get_invoice", "get_refund",
        "newsletter_subscription", "payment_issue", "place_order", "recover_password",
        "registration_problems", "review", "set_up_shipping_address", "switch_account",
        "track_order", "track_refund"
    ],
    "Account Change": ["switch_account"],
    "Program": ["place_order"],
    "Refund Amount": ["track_refund"],
    "Money Amount": ["check_refund_policy", "complaint", "get_refund", "track_refund"],
    "Store Location": ["complaint", "delivery_options", "place_order"]
}

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

def load_and_split_dataset(dataset_path, dataset_size=None):
    df = pd.read_csv(dataset_path)

    if dataset_size is None:
        df_small = df
    else:
        df_small, _ = train_test_split(
            df, train_size=dataset_size, stratify=df['category'], random_state=42
        )

    df_train, df_temp = train_test_split(
        df_small, test_size=0.2, stratify=df_small['category'], random_state=42
    )

    df_val, df_test = train_test_split(
        df_temp, test_size=0.5, stratify=None, random_state=42
    )

    return df_train, df_val, df_test

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

def model_loader(model_id: str):
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )

    tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        quantization_config=bnb_config,
        trust_remote_code=True
    )

    generator = pipeline(task="text-generation", model=model, tokenizer=tokenizer)
    return model, tokenizer, generator

In [ ]:
def general_prompting_tinyllama(df_test, generator, categories):
    correct = 0
    total = len(df_test)
    predictions = []

    system_prompt = {
        "role": "system",
        "content": (
            "You are an AI assistant that classifies user support queries into one of the following categories:\n\n"
            f"{chr(10).join(f'- {c}' for c in categories)}\n\n"
            "Your task is to read the user's query and return only the most relevant category name from the list above.\n"
            "Do not provide explanations or additional text—only output the category name."
        )
    }

    for step, (_, row) in enumerate(df_test.iterrows(), 1):
        user_prompt = {
            "role": "user",
            "content": row["instruction"]
        }

        assistant_prompt = {
            "role": "assistant",
            "content": "Labeled Category:"
        }

        messages = [system_prompt, user_prompt, assistant_prompt]
        prompt = "\n".join(f"<|{msg['role']}|>\n{msg['content']}" for msg in messages)

        output = generator(prompt, max_new_tokens=10)[0]["generated_text"]

        predicted_category = "UNKNOWN"
        for cat in categories:
            if cat in output:
                predicted_category = cat
                break

        actual_category = row["category"]
        is_correct = predicted_category == actual_category
        correct += is_correct
        predictions.append({
            "instruction": row["instruction"],
            "actual": actual_category,
            "predicted": predicted_category,
            "correct": is_correct
        })

    accuracy = correct / total * 100

    return predictions, total, correct, accuracy

In [ ]:
import torch
from torch.optim import AdamW
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset

def generate_input_output_pair_tinyllama(df_train, tokenizer, max_length=512):
    prompts = [
        [
            {"role": "user", "content": instruction},
            {"role": "assistant", "content": "Labeled Category:"}
        ]
        for instruction in df_train['instruction']
    ]
    responses = df_train['category'].tolist()

    chat_templates = tokenizer.apply_chat_template(prompts, continue_final_message=True, tokenize=False)
    full_response_text = [
        (chat_template + " " + target_response + tokenizer.eos_token)
        for chat_template, target_response in zip(chat_templates, responses)
    ]

    input_ids_tokenized = tokenizer(
        full_response_text,
        return_tensors="pt",
        add_special_tokens=False,
        padding="max_length",
        max_length=max_length,
        truncation=True
    )["input_ids"]

    labels_tokenized = tokenizer(
        [" " + response + tokenizer.eos_token for response in responses],
        add_special_tokens=False,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=input_ids_tokenized.shape[1]
    )["input_ids"]

    labels_tokenized_fixed = torch.where(labels_tokenized != tokenizer.pad_token_id, labels_tokenized, -100)
    labels_tokenized_fixed[:, -1] = -100

    input_ids_tokenized_left_shifted = input_ids_tokenized[:, :-1]
    labels_tokenized_right_shifted = labels_tokenized_fixed[:, 1:]

    attention_mask = input_ids_tokenized_left_shifted != tokenizer.pad_token_id

    return {
        "input_ids": input_ids_tokenized_left_shifted,
        "attention_mask": attention_mask,
        "labels": labels_tokenized_right_shifted
    }

def custom_finetune_tinyllama(df_train, model, tokenizer, data, batch_size=1, lr=1e-5, decay=0.01, epochs=3):
    dataset = TensorDataset(data["input_ids"], data["attention_mask"], data["labels"])
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model.train()

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=decay)

    print("input_ids max:", data["input_ids"].max().item())
    print("labels max:", data["labels"].max().item())
    print("attention_mask shape:", data["attention_mask"].shape)

    for epoch in range(epochs):
        for input_ids, attention_mask, labels in tqdm(loader, desc=f"Epoch {epoch+1}"):
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

            print(outputs)
            
            loss = outputs.loss

            if torch.isnan(loss):
               print("⚠️ NaN loss — skipping this batch")
               continue

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            print(f"Loss: {loss.item():.4f}")

In [ ]:
!pip install -U transformers peft bitsandbytes accelerate datasets
import re
from datasets import Dataset
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

def format_chat(example, tokenizer):
    example["text"] = f"Instruction: {example['instruction']}\nLabeled Category: {example['category']}{tokenizer.eos_token}"
    return example

def tokenize_function(example, tokenizer, max_length=512):
    tokenized = tokenizer(example["text"], padding="max_length", truncation=True, max_length=max_length)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

def finetune_train_tinyllama(df_train, model, tokenizer):
    dataset = Dataset.from_pandas(df_train[["instruction", "category"]])
    dataset = dataset.map(lambda x: format_chat(x, tokenizer))
    
    dataset = dataset.map(lambda x: tokenize_function(x, tokenizer), batched=True)
    dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    training_args = TrainingArguments(
        output_dir="./tinyllama-lora-supportbot",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        num_train_epochs=3,
        logging_steps=10,
        save_strategy="epoch",
        report_to="none",
        fp16=True,
        label_names=["labels"]
    )

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        tokenizer=tokenizer,
        data_collator=data_collator
    )

    trainer.train()


def predict_output(text):
    prompt = f"Instruction: {text}\nLabeled Category:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        temperature=0.0,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_output

def extract_category(text, categories):
    match = re.search(r"Labeled Category:\s*(.*)", text, re.IGNORECASE)
    if match:
        label_section = match.group(1).strip().upper()

        categories_upper = [cat.upper() for cat in categories]

        for cat in sorted(categories_upper, key=len, reverse=True):
            if cat in label_section:
                return cat

    return "NOT FOUND"

def evaluate_model_on_test_set_tinyllama(df_test, categories):
    predictions = []
    correct = 0
    total = len(df_test)

    for _, row in df_test.iterrows():
        instruction = row["instruction"]
        true_category = row["category"].strip().upper()

        pred = predict_output(instruction)
        pred_category = extract_category(pred, categories)
        pred_category = pred_category.strip().upper()
        predictions.append(pred)

        print("Predicted text: ", pred)
        print("Predicted Category: ", pred_category)
        print("Correct Category: ", true_category)
        print("")

        if pred_category == true_category:
            correct += 1

    accuracy = (correct / total) * 100

    return predictions, correct, total, accuracy

In [ ]:
import warnings
warnings.filterwarnings("ignore")

dataset_path = "path_to_dataset"
model_id = "Doctor-Shotgun/TinyLlama-1.1B-32k-Instruct"
dataset_size = 500

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

df_train, df_val, df_test = load_and_split_dataset(dataset_path, dataset_size)

model, tokenizer, generator = model_loader(model_id)
model.gradient_checkpointing_disable()

predictions, total, correct, accuracy = general_prompting_tinyllama(df_test, generator, categories)
print(f"Correct with General prompting: {correct}")
print(f"Total with General prompting: {total}")
print(f"Accuracy with General prompting: {accuracy:.2f}%")

finetune_train_tinyllama(df_train, model, tokenizer)
predictions, correct, total, accuracy = evaluate_model_on_test_set_tinyllama(df_test, categories)
print(f"Correct with LoRa fine-tuning: {correct}")
print(f"Total with LoRa fine-tuning: {total}")
print(f"Accuracy with LoRa fine-tuning: {accuracy:.2f}%")

custom_batch_size = 4
custom_lr = 1e-5
custom_decay = 0.01
custom_epochs = 5
custom_pair_data = generate_input_output_pair_tinyllama(df_train, tokenizer)
custom_finetune_tinyllama(
    df_train, model, tokenizer, custom_pair_data,
    device, custom_batch_size, custom_lr, custom_decay, custom_epochs
)